In [12]:
import numpy as np
import pandas as pd
import pyarrow as pa
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

import sys
from pathlib import Path
import os
from dotenv import load_dotenv

pd.set_option("display.max_columns", None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)


In [2]:
import sys
from pathlib import Path

# Add the project root directory to sys.path
# (Adjust .parent counts depending on how deep your notebook is)
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Now import directly from your folder structure
from scripts.data_cleaning import clean_data, add_columns

In [3]:
df = pd.read_parquet(f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet")

In [5]:
df.shape

(3066766, 19)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3066766 entries, 0 to 3066765
Data columns (total 19 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int64         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int64         
 8   DOLocationID           int64         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  airport_fee           

In [13]:
df.describe()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
count,3066766.000,3066766,3066766,2995023.000,3066766.000,2995023.000,3066766.000,3066766.000,3066766.000,3066766.000,3066766.000,3066766.000,3066766.000,3066766.000,3066766.000,3066766.000,2995023.000,2995023.000
mean,1.730,2023-01-17 00:22:26.288164,2023-01-17 00:38:06.427874,1.363,3.847,1.497,166.398,164.393,1.194,18.367,1.538,0.488,3.368,0.518,0.982,27.020,2.274,0.107
min,1.000,2008-12-31 23:01:42,2009-01-01 14:29:11,0.000,0.000,1.000,1.000,1.000,0.000,-900.000,-7.500,-0.500,-96.220,-65.000,-1.000,-751.000,-2.500,-1.250
25%,1.000,2023-01-09 16:21:57.250000,2023-01-09 16:37:06,1.000,1.060,1.000,132.000,114.000,1.000,8.600,0.000,0.500,1.000,0.000,1.000,15.400,2.500,0.000
50%,2.000,2023-01-17 08:42:29.500000,2023-01-17 08:58:30.500000,1.000,1.800,1.000,162.000,162.000,1.000,12.800,1.000,0.500,2.720,0.000,1.000,20.160,2.500,0.000
75%,2.000,2023-01-24 16:26:27,2023-01-24 16:42:49,1.000,3.330,1.000,234.000,234.000,1.000,20.500,2.500,0.500,4.200,0.000,1.000,28.700,2.500,0.000
max,2.000,2023-02-01 00:56:53,2023-02-02 09:28:47,9.000,258928.150,99.000,265.000,265.000,4.000,1160.100,12.500,53.160,380.800,196.990,1.000,1169.400,2.500,1.250
std,0.444,NaN,NaN,0.896,249.584,6.475,64.244,69.944,0.529,17.808,1.790,0.103,3.827,2.018,0.183,22.164,0.772,0.356


In [8]:
df.isna().sum()

VendorID                     0
tpep_pickup_datetime         0
tpep_dropoff_datetime        0
passenger_count          71743
trip_distance                0
RatecodeID               71743
store_and_fwd_flag       71743
PULocationID                 0
DOLocationID                 0
payment_type                 0
fare_amount                  0
extra                        0
mta_tax                      0
tip_amount                   0
tolls_amount                 0
improvement_surcharge        0
total_amount                 0
congestion_surcharge     71743
airport_fee              71743
dtype: int64

In [22]:
df['VendorID'].value_counts()

VendorID
2    2239399
1     827367
Name: count, dtype: int64

In [4]:
df_clean = clean_data(df, 1)[0]

In [9]:
df_clean.shape

(2883039, 29)

In [10]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2883039 entries, 0 to 2995022
Data columns (total 29 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   VendorID                int64         
 1   tpep_pickup_datetime    datetime64[us]
 2   tpep_dropoff_datetime   datetime64[us]
 3   passenger_count         float64       
 4   trip_distance           float64       
 5   RatecodeID              float64       
 6   store_and_fwd_flag      object        
 7   PULocationID            int64         
 8   DOLocationID            int64         
 9   payment_type            int64         
 10  fare_amount             float64       
 11  extra                   float64       
 12  mta_tax                 float64       
 13  tip_amount              float64       
 14  tolls_amount            float64       
 15  improvement_surcharge   float64       
 16  total_amount            float64       
 17  congestion_surcharge    float64       
 18  airport

In [14]:
df_clean.describe()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,abs_fare,abs_total,pickup_month,pickup_day,pickup_hour,pickup_day_of_week_num,trip_duration_minutes
count,2883039.000,2883039,2883039,2883039.000,2883039.000,2883039.000,2883039.000,2883039.000,2883039.000,2883039.000,2883039.000,2883039.000,2883039.000,2883039.000,2883039.000,2883039.000,2883039.000,2883039.000,2883039.000,2883039.000,2883039.000,2883039.000,2883039.000,2883039.000,2883039.000
mean,1.742,2023-01-17 00:41:01.673194,2023-01-17 00:56:44.321236,1.388,3.404,1.453,166.524,164.538,1.188,18.630,1.571,0.496,3.428,0.529,0.999,27.460,2.317,0.110,18.630,27.460,1.000,16.417,14.170,3.003,15.711
min,1.000,2023-01-01 00:00:05,2023-01-01 00:03:28,1.000,0.000,1.000,1.000,1.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,1.000,0.000,0.000,0.000
25%,1.000,2023-01-09 16:39:15.500000,2023-01-09 16:54:53.500000,1.000,1.090,1.000,132.000,114.000,1.000,8.600,0.000,0.500,1.000,0.000,1.000,15.480,2.500,0.000,8.600,15.480,1.000,9.000,11.000,1.000,7.150
50%,2.000,2023-01-17 08:49:18,2023-01-17 09:05:26,1.000,1.800,1.000,162.000,162.000,1.000,12.800,1.000,0.500,2.800,0.000,1.000,20.160,2.500,0.000,12.800,20.160,1.000,17.000,15.000,3.000,11.500
75%,2.000,2023-01-24 16:26:57,2023-01-24 16:43:20,1.000,3.340,1.000,234.000,234.000,1.000,20.500,2.500,0.500,4.200,0.000,1.000,28.700,2.500,0.000,20.500,28.700,1.000,24.000,19.000,5.000,18.250
max,2.000,2023-01-31 23:59:59,2023-02-01 23:18:41,9.000,96.700,99.000,265.000,265.000,4.000,999.000,12.500,53.160,380.800,196.990,1.000,1000.000,2.500,1.250,999.000,1000.000,1.000,31.000,23.000,6.000,1439.800
std,0.438,NaN,NaN,0.889,4.415,6.153,64.040,69.884,0.412,17.366,1.778,0.053,3.844,2.017,0.031,21.684,0.651,0.355,17.366,21.684,0.000,8.694,5.755,1.990,42.634


In [15]:
2883039 / 3066766

0.9400909622710047

In [18]:
df_clean[df_clean['trip_distance'] == 0].head(15)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,abs_fare,abs_total,pickup_month,pickup_day,pickup_hour,pickup_day_of_week_num,pickup_day_of_week,pickup_date,is_weekend,trip_duration_minutes
333,1,2023-01-01 00:57:44,2023-01-01 00:57:59,1.000,0.000,1.000,N,137,137,3,3.000,3.500,0.500,0.000,0.000,1.000,8.000,2.500,0.000,3.000,8.000,1,1,0,6,Sunday,2023-01-01,True,0.250
398,2,2023-01-01 00:28:04,2023-01-01 00:28:35,1.000,0.000,2.000,N,142,142,2,70.000,0.000,0.500,0.000,0.000,1.000,74.000,2.500,0.000,70.000,74.000,1,1,0,6,Sunday,2023-01-01,True,0.517
996,2,2023-01-01 00:16:54,2023-01-01 00:17:00,1.000,0.000,1.000,N,197,197,2,3.000,1.000,0.500,0.000,0.000,1.000,5.500,0.000,0.000,3.000,5.500,1,1,0,6,Sunday,2023-01-01,True,0.100
1390,2,2023-01-01 00:25:04,2023-01-01 00:29:16,2.000,0.000,1.000,N,50,50,2,3.000,1.000,0.500,0.000,0.000,1.000,8.000,2.500,0.000,3.000,8.000,1,1,0,6,Sunday,2023-01-01,True,4.200
2380,2,2023-01-01 00:38:40,2023-01-01 00:39:00,1.000,0.000,5.000,N,237,237,1,12.000,0.000,0.000,0.000,0.000,1.000,15.500,2.500,0.000,12.000,15.500,1,1,0,6,Sunday,2023-01-01,True,0.333
2464,2,2023-01-01 00:11:21,2023-01-01 00:11:25,1.000,0.000,2.000,N,164,164,2,70.000,0.000,0.500,0.000,0.000,1.000,71.500,0.000,0.000,70.000,71.500,1,1,0,6,Sunday,2023-01-01,True,0.067
2499,2,2023-01-01 00:54:56,2023-01-01 00:55:04,2.000,0.000,5.000,N,265,265,1,110.000,0.000,0.000,22.200,0.000,1.000,133.200,0.000,0.000,110.000,133.200,1,1,0,6,Sunday,2023-01-01,True,0.133
2679,2,2023-01-01 00:03:08,2023-01-01 00:03:28,1.000,0.000,1.000,N,70,70,2,3.000,1.000,0.500,0.000,0.000,1.000,5.500,0.000,0.000,3.000,5.500,1,1,0,6,Sunday,2023-01-01,True,0.333
2722,2,2023-01-01 00:48:12,2023-01-01 00:48:25,1.000,0.000,5.000,N,87,87,1,27.000,0.000,0.000,6.100,0.000,1.000,36.600,2.500,0.000,27.000,36.600,1,1,0,6,Sunday,2023-01-01,True,0.217
2929,1,2023-01-01 00:22:50,2023-01-01 00:23:58,2.000,0.000,1.000,N,48,48,1,3.000,3.500,0.500,1.600,0.000,1.000,9.600,2.500,0.000,3.000,9.600,1,1,0,6,Sunday,2023-01-01,True,1.133


In [19]:
df_clean[df_clean['trip_distance'] == 0].tail(15)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,abs_fare,abs_total,pickup_month,pickup_day,pickup_hour,pickup_day_of_week_num,pickup_day_of_week,pickup_date,is_weekend,trip_duration_minutes
2992759,2,2023-01-31 23:20:20,2023-01-31 23:20:28,1.000,0.000,5.000,N,125,125,1,78.300,0.000,0.000,16.360,0.000,1.000,98.160,2.500,0.000,78.300,98.160,1,31,23,1,Tuesday,2023-01-31,False,0.133
2993486,2,2023-01-31 23:40:57,2023-01-31 23:41:02,1.000,0.000,5.000,N,141,141,1,5.000,0.000,0.000,1.700,0.000,1.000,10.200,2.500,0.000,5.000,10.200,1,31,23,1,Tuesday,2023-01-31,False,0.083
2993521,2,2023-01-31 23:12:12,2023-01-31 23:12:22,1.000,0.000,5.000,N,231,231,1,13.000,0.000,0.000,3.300,0.000,1.000,19.800,2.500,0.000,13.000,19.800,1,31,23,1,Tuesday,2023-01-31,False,0.167
2993648,2,2023-01-31 23:22:08,2023-01-31 23:22:16,1.000,0.000,1.000,N,132,132,2,3.000,1.000,0.500,0.000,0.000,1.000,6.750,0.000,1.250,3.000,6.750,1,31,23,1,Tuesday,2023-01-31,False,0.133
2993703,2,2023-01-31 23:13:37,2023-01-31 23:13:55,2.000,0.000,5.000,N,265,265,1,344.000,0.000,0.000,69.000,0.000,1.000,414.000,0.000,0.000,344.000,414.000,1,31,23,1,Tuesday,2023-01-31,False,0.300
2993772,2,2023-01-31 23:28:05,2023-01-31 23:28:27,1.000,0.000,2.000,N,231,231,1,70.000,0.000,0.500,5.000,0.000,1.000,80.250,2.500,1.250,70.000,80.250,1,31,23,1,Tuesday,2023-01-31,False,0.367
2993880,2,2023-01-31 23:26:12,2023-01-31 23:26:19,1.000,0.000,5.000,N,236,236,1,16.400,0.000,0.000,5.970,0.000,1.000,25.870,2.500,0.000,16.400,25.870,1,31,23,1,Tuesday,2023-01-31,False,0.117
2994143,2,2023-01-31 23:01:59,2023-01-31 23:02:09,1.000,0.000,1.000,N,132,132,2,3.000,1.000,0.500,0.000,0.000,1.000,6.750,0.000,1.250,3.000,6.750,1,31,23,1,Tuesday,2023-01-31,False,0.167
2994301,2,2023-01-31 23:21:31,2023-01-31 23:21:38,3.000,0.000,5.000,N,230,230,1,33.330,0.000,0.500,7.470,0.000,1.000,44.800,2.500,0.000,33.330,44.800,1,31,23,1,Tuesday,2023-01-31,False,0.117
2994643,1,2023-01-31 23:24:48,2023-01-31 23:25:06,1.000,0.000,1.000,N,237,237,3,3.000,3.500,0.500,0.000,0.000,1.000,8.000,2.500,0.000,3.000,8.000,1,31,23,1,Tuesday,2023-01-31,False,0.300
